In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Load data
train_df = pd.read_csv("../data/dataset/processed/go_emotion_dutch.csv")
test_df = pd.read_csv("../data/group_4_url_1_transcript.csv")

# Prepare text
train_texts = train_df['Sentence'].astype(str).tolist()
test_texts = test_df['Sentence'].astype(str).tolist()

# Tokenize
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts + test_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

# Pad
MAX_LENGTH = 30
train_padded = pad_sequences(train_sequences, maxlen=MAX_LENGTH, padding='post')
test_padded = pad_sequences(test_sequences, maxlen=MAX_LENGTH, padding='post')

# Labels
label_encoder = LabelEncoder()
label_encoder.fit(train_df['Emotion_core'].tolist() + test_df['Emotion_core'].tolist())

train_labels = label_encoder.transform(train_df['Emotion_core'].tolist())
test_labels = label_encoder.transform(test_df['Emotion_core'].tolist())

num_classes = len(label_encoder.classes_)
train_categorical = to_categorical(train_labels, num_classes)
test_categorical = to_categorical(test_labels, num_classes)

print(f"Vocab size: {len(tokenizer.word_index)}")
print(f"Classes: {num_classes}")

# Build model
model = Sequential([
    Embedding(len(tokenizer.word_index) + 1, 64, input_length=MAX_LENGTH),
    LSTM(32),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.build(input_shape=(None, MAX_LENGTH))
print("\nModel:")
model.summary()

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Train
history = model.fit(
    train_padded, train_categorical,
    validation_data=(test_padded, test_categorical),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Evaluate
predictions = model.predict(test_padded)
predicted_classes = np.argmax(predictions, axis=1)

accuracy = accuracy_score(test_labels, predicted_classes)
f1 = f1_score(test_labels, predicted_classes, average='weighted')

print(f"\nAccuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, predicted_classes, target_names=label_encoder.classes_))

Vocab size: 40491
Classes: 7

Model:


/Users/florislokhorst/Desktop/School/Y2_A/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 30, 64)         │     2,591,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,604,135 (9.93 MB)

 Trainable params: 2,604,135 (9.93 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.4318 - loss: 1.4720 - val_accuracy: 0.5486 - val_loss: 1.4967 - learning_rate: 0.0010
Epoch 2/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.5094 - loss: 1.3048 - val_accuracy: 0.5495 - val_loss: 1.4154 - learning_rate: 0.0010
Epoch 3/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.5430 - loss: 1.2232 - val_accuracy: 0.5352 - val_loss: 1.3999 - learning_rate: 0.0010
Epoch 4/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.5789 - loss: 1.1487 - val_accuracy: 0.5248 - val_loss: 1.4089 - learning_rate: 0.0010
Epoch 5/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.6039 - loss: 1.0869 - val_accuracy: 0.4962 - val_loss: 1.4289 - learning_rate: 0.0010
Epoch 6/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.6247 - loss: 1.0307 - val_accuracy: 0.4981 - val_loss: 1.4739 - learning_rate: 0.0010
Epoch 7/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.6460 - 

/Users/florislokhorst/Desktop/School/Y2_A/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/florislokhorst/Desktop/School/Y2_A/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/florislokhorst/Desktop/School/Y2_A/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 